# Smart Vitals: your first model
**CMS22301 Intelligent Systems and Emerging Technologies · Project starter · Ravensbourne University London**

Your assessment brief asks for *"a trained and evaluated machine learning model"* that classifies each reading as **Normal, Warning or Critical**, evaluated with *"accuracy, precision, recall, F1-score, confusion matrix"*. This notebook builds the first version of that model, so the rest of your system has something real to connect to.

By the end you will have:
- a **baseline**: the simplest possible answer, which your model must beat
- a **first model**, tested on **patients it has never seen**, with the measures the brief asks for
- the model **saved** in `models/`, ready for the Week 8 API
- a **results file** in `docs/results/` and the evidence for **ADR 1**

**How to run:** Jupyter (`Shift + Enter` runs a cell) or Google Colab (File → Upload notebook). Run the cells **in order from the top**.
Lines marked `# TODO` contain `___` blanks for you to complete. The CHECK cells print **PASS**, **TODO** (not finished yet) or **FIX** (with a hint).
If something breaks: *Kernel → Restart & Run All* (Jupyter) or *Runtime → Run all* (Colab).

**Data:** every step runs on **demo data** (simulated readings from 60 patients) until you point the notebook at your own simulator's output or an open dataset. **Never use real patient data** (brief, requirement 7). About 1 hour.

## Setup
Run this cell first. It loads the tools, finds your project folder and creates the demo data if it is missing.

In [ ]:
# SETUP: run this cell first. You do not need to change it.
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, ConfusionMatrixDisplay)

pd.set_option("display.width", 130); pd.set_option("display.max_columns", 30)

# Find the project folder, whether this runs from notebooks/ (Jupyter) or on its own (Colab)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for folder in ["data", "models", "docs/results"]:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
DEMO_PATH = ROOT / "data" / "demo_vitals.csv"
ORDER = ["Normal", "Warning", "Critical"]


def make_demo_data(patients=60, readings=30, seed=42):
    """Simulated Smart Vitals readings for practice. One row = one reading from one patient.
    The Normal / Warning / Critical labels come from a points rule INVENTED FOR TEACHING:
    it is not a clinical score. No real patient data is used."""
    rs = np.random.RandomState(seed)
    n = patients * readings
    pid = np.repeat([f"P{p:03d}" for p in range(1, patients + 1)], readings)
    risk = np.repeat(rs.choice([1, 2, 3], size=patients, p=[.5, .3, .2]), readings)        # 1 Low, 2 Medium, 3 High
    diabetic = np.repeat((rs.rand(patients) < 0.35).astype(int), readings)
    base_hr = np.repeat(rs.normal(72, 7, patients), readings)
    base_sys = np.repeat(rs.normal(118, 9, patients), readings) + 6 * risk
    unwell = rs.rand(n) < (0.10 * risk + 0.06)                    # higher risk, more unwell readings
    k = np.where(unwell, rs.uniform(0.15, 1.5, n), 0.0)            # how unwell
    heart = base_hr + rs.normal(0, 6, n) + k * rs.choice([-22, 38], n)
    spo2 = np.minimum(100, rs.normal(97, 1.1, n) - k * rs.uniform(2, 7, n))
    systolic = base_sys + rs.normal(0, 8, n) + k * rs.choice([-35, 55], n)
    diastolic = 0.55 * systolic + rs.normal(12, 5, n)
    temp = rs.normal(36.8, 0.3, n) + k * rs.uniform(0, 2.2, n)
    glucose = np.where(diabetic == 1, rs.normal(135, 25, n) + k * rs.choice([-60, 150], n), np.nan)
    points = (np.where((heart < 50) | (heart > 120), 2, np.where(heart > 100, 1, 0))
              + np.where(spo2 < 91, 3, np.where(spo2 < 94, 1, 0))
              + np.where((systolic < 90) | (systolic > 200), 3, np.where((systolic < 100) | (systolic > 170), 1, 0))
              + np.where((temp > 39) | (temp < 35.5), 2, np.where(temp > 38, 1, 0))
              + np.where(np.isnan(glucose), 0, np.where((glucose < 70) | (glucose > 300), 3,
                                                        np.where(glucose > 220, 1, 0))))
    status = np.where(points >= 3, "Critical", np.where(points >= 1, "Warning", "Normal"))
    noisy = rs.rand(n) < 0.02                                     # real labels are never perfect
    status = np.where(noisy, rs.choice(ORDER, n), status)
    df = pd.DataFrame({"patient_id": pid, "risk_level": risk, "diabetic": diabetic,
                       "heart_rate": heart.round(), "spo2": spo2.round(1), "systolic_bp": systolic.round(),
                       "diastolic_bp": diastolic.round(), "temperature": temp.round(1),
                       "glucose": np.round(glucose), "status": status})
    for col in ["heart_rate", "spo2", "systolic_bp", "temperature"]:      # sensor dropout: 2% missing
        df.loc[rs.rand(n) < 0.02, col] = np.nan
    return df


if not DEMO_PATH.exists():
    make_demo_data().to_csv(DEMO_PATH, index=False)

STEPS = {}


def short(problem, limit=200):
    """Keep error messages to one readable line."""
    text = " ".join(str(problem).split())
    return text if len(text) <= limit else text[:limit] + " ..."


def check(name, test):
    """Run one test and say PASS, TODO or FIX."""
    try:
        test()
        print(f"PASS  {name}")
    except NameError as problem:
        if "'___'" in str(problem):
            print(f"TODO  {name}: replace every ___ with your code")
        else:
            print(f"FIX   {name}: {problem} (run the cells above in order, and check the spelling)")
    except AssertionError as problem:
        message = str(problem)
        print(message if message.startswith("TODO") else f"FIX   {name}: {message}")
    except Exception as problem:
        print(f"ERROR {name}: {type(problem).__name__}: {short(problem)}")


def attempt(step, action):
    """Run one step; print TODO or FIX instead of crashing."""
    STEPS[step] = False
    try:
        result = action()
    except NameError as problem:
        if "'___'" in str(problem):
            print(f"TODO  {step}: replace every ___ in the cell above, then run it again")
        else:
            print(f"FIX   {step}: {problem} (run the cells above in order, and check the spelling)")
        return None
    except Exception as problem:
        print(f"FIX   {step}: {type(problem).__name__}: {short(problem)}")
        return None
    STEPS[step] = True
    return result


def needs(*steps):
    """True when the earlier steps are finished."""
    missing = [s for s in steps if not STEPS.get(s)]
    if missing:
        print("TODO  finish these steps first:", ", ".join(missing))
    return not missing


def scores(y_true, y_pred):
    """The measures the brief asks for, plus recall for Critical: a missed Critical is a missed alert."""
    s = {"accuracy": accuracy_score(y_true, y_pred),
         "balanced accuracy": balanced_accuracy_score(y_true, y_pred),
         "precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
         "recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
         "F1 (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0)}
    if "Critical" in set(y_true):
        s["recall: Critical"] = recall_score(y_true, y_pred, labels=["Critical"], average="macro", zero_division=0)
    return s


print("Setup done. Project folder:", ROOT)
print("Demo data:", DEMO_PATH.relative_to(ROOT), "| rows:", len(pd.read_csv(DEMO_PATH)))

## Step 1 · Your model in one line
What exactly does your model decide, from what, and how will you know it is good enough? In Step 7 you compare your results with this target, so be specific.

In [ ]:
MY_MODEL = {
    "decides": "",       # TODO: e.g. "is this reading Normal, Warning or Critical?"
    "inputs": "",        # TODO: e.g. "the five vital signs, risk level and diabetes status"
    "main_measure": "",  # TODO: e.g. "recall for Critical: a missed Critical is a missed alert"
    "good_enough": "",   # TODO: your target, e.g. "recall for Critical of at least 0.90"
}

In [ ]:
# CHECK
def test_model_plan():
    empty = [k for k, v in MY_MODEL.items() if not str(v).strip()]
    assert not empty, "TODO  your model in one line: fill in " + ", ".join(empty)

check("your model in one line", test_model_plan)

## Step 2 · Load your data
### 2a · Worked example: the demo data
One row is one reading from one patient: five vital signs (glucose only for patients with diabetes), the patient's risk level (1 Low, 2 Medium, 3 High) and a label, `status`. About 2% of readings are missing, as if a sensor dropped out.

The labels come from a points rule **invented for teaching**, loosely in the spirit of hospital early warning scores such as NEWS2 (Royal College of Physicians, 2017). It is **not** a clinical score.

In [ ]:
demo = pd.read_csv(DEMO_PATH)
print("Rows and columns:", demo.shape, "| patients:", demo["patient_id"].nunique())
print(demo.head())
print()
print("What we predict, status:")
print(demo["status"].value_counts())

### 2b · Your turn: point the notebook at your data
When your own simulator (Weeks 4 and 6) or an open, anonymised dataset is ready, save it as a CSV in `data/` and change `DATA_PATH`. **Until then, keep the demo settings and carry on.**

- `TARGET`: the label column (Normal, Warning or Critical).
- `DROP`: columns the model must **not** see, such as the patient ID, or anything written down *after* the label was known. A column that gives the answer away makes a model look brilliant here and fail in use.

In [ ]:
DATA_PATH = DEMO_PATH            # TODO: your own simulated or open dataset, e.g. ROOT / "data" / "my_vitals.csv"
TARGET = "status"                # the column your model predicts: Normal, Warning or Critical
DROP = ["patient_id"]            # columns the model must not see

df = pd.read_csv(DATA_PATH)
blank = df[TARGET].isna().sum()
df = df.dropna(subset=[TARGET]).reset_index(drop=True)   # rows without a label cannot teach anything
features = df.drop(columns=[TARGET] + DROP)
X = features.select_dtypes("number")                    # this starter uses number columns only
y = df[TARGET]
left_out = [c for c in features.columns if c not in X.columns]
print("Rows:", len(df), "| feature columns used:", list(X.columns))
if blank:
    print("Rows with no label, removed:", blank)
if left_out:
    print("Not numbers, so left out for now:", left_out)

In [ ]:
# CHECK
def test_loaded():
    assert len(df) >= 50, f"only {len(df)} rows: you need more data for a fair test (aim for hundreds)"
    assert X.shape[1] >= 1, "no number columns left to learn from"

def test_labels():
    assert y.nunique() <= 20, f"{y.nunique()} different labels: is TARGET the right column?"
    assert y.value_counts().min() >= 5, "some labels appear fewer than 5 times: the model cannot learn them"

def test_own_data():
    assert Path(DATA_PATH) != DEMO_PATH, ("TODO  your own data: still the demo data "
                                          "(fine for practice; switch when your simulator or dataset is ready)")

check("data loaded", test_loaded)
check("labels look right", test_labels)
check("your own data", test_own_data)

## Step 3 · A data health check
How big is the data, what is missing, and how balanced are the labels? Write anything surprising in your data card, `data/DATA_CARD.md`.

In [ ]:
print(f"Rows: {len(df)}   Feature columns: {X.shape[1]}")
missing = X.isna().sum()
print("Missing values:", missing[missing > 0].to_dict() or "none")
share = y.value_counts(normalize=True)
print("Labels:", {k: f"{v:.1%}" for k, v in share.items()})
print(f"Always answering '{share.index[0]}' would be right {share.iloc[0]:.0%} of the time.")

**Think:** Which values are missing, and why? (Glucose is missing for every patient without diabetes: is that the same kind of gap as a sensor dropout?) How rare is Critical, and what does that mean for accuracy?

**Your answer:**

_(double-click to type)_

## Step 4 · Keep back a test set of unseen patients
A model must be tested on data it has **never seen**. For patient monitoring, that means **patients** it has never seen: readings from the same person look alike, so a random split of rows lets the model "recognise" a patient instead of reading the vitals. On the demo data, a random split makes the forest's recall for Critical look like 0.91; tested on unseen patients it is 0.81. The first number is too kind.

### 4a · Worked example

In [ ]:
# Worked example: 4 patients, 2 readings each. Keep back 25% of the PATIENTS, not of the rows.
readings = pd.DataFrame({"patient": ["A", "A", "B", "B", "C", "C", "D", "D"],
                         "heart_rate": [70, 72, 95, 99, 60, 62, 110, 115]})
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
learn, test = next(splitter.split(readings, groups=readings["patient"]))
print("Learn from patients:", sorted(set(readings["patient"].iloc[learn])))
print("Test on patients:   ", sorted(set(readings["patient"].iloc[test])))

### 4b · Your turn
Replace the one `___`, run the cell, then run the CHECK cell.

In [ ]:
GROUP = "patient_id"             # test on patients the model never saw (None: split rows at random)

def split(X, y):
    """Keep back a test set the model never sees."""
    test_share = ___             # TODO: the share kept back for testing (the worked example used 0.25; 0.2 is common)
    if GROUP:
        assert GROUP in df.columns, f"there is no '{GROUP}' column: set GROUP to your patient ID column, or to None"
        splitter = GroupShuffleSplit(n_splits=1, test_size=test_share, random_state=42)
        learn, test = next(splitter.split(X, y, groups=df[GROUP]))
        return X.iloc[learn], X.iloc[test], y.iloc[learn], y.iloc[test]
    return train_test_split(X, y, test_size=test_share, random_state=42, stratify=y)

parts = attempt("split", lambda: split(X, y))
if parts:
    X_train, X_test, y_train, y_test = parts
    print(f"Learn from {len(X_train)} rows, test on {len(X_test)} rows")

In [ ]:
# CHECK
def test_split():
    a, b, c, d = split(X, y)
    share = len(b) / len(X)
    assert 0.1 <= share <= 0.4, f"the test set is {share:.0%} of the rows: keep back between 10% and 40%"
    assert not set(a.index) & set(b.index), "some rows are in both parts"
    if GROUP:
        assert not set(df.loc[a.index, GROUP]) & set(df.loc[b.index, GROUP]), "a patient is in both parts"

check("test set kept back", test_split)

## Step 5 · The baseline: the answer to beat
A **baseline** is the simplest possible answer, with no learning at all: always give the most common label. If your model cannot beat it, it has learned nothing useful, however good its score looks.

### 5a · Worked example

In [ ]:
# Worked example: 100 readings: 70 Normal, 20 Warning, 10 Critical
truth = pd.Series(["Normal"] * 70 + ["Warning"] * 20 + ["Critical"] * 10)
nothing = np.zeros((100, 1))                       # the baseline does not even look at the readings
always = DummyClassifier(strategy="most_frequent").fit(nothing, truth)
guess = always.predict(nothing)
print("The baseline always says:", guess[0])
for name, value in scores(truth, guess).items():
    print(f"{name:<18} {value:.2f}")

Accuracy is 0.70, yet the baseline never finds a single Critical reading: **recall for Critical is 0**, so no clinician would ever be alerted. **Balanced accuracy** (the average hit rate across the three labels) is 0.33. When one label is rare and important, judge the model on recall for that label and on balanced accuracy, not on accuracy.

### 5b · Your turn
Replace the one `___`.

In [ ]:
def make_baseline():
    """The simplest answer, with no learning at all."""
    return DummyClassifier(strategy=___)        # TODO: copy the strategy from the worked example

if needs("split"):
    baseline = attempt("baseline", lambda: make_baseline().fit(X_train, y_train))
    if STEPS["baseline"]:
        print("Baseline ready. It always answers:", baseline.predict(X_test.iloc[[0]])[0])

In [ ]:
# CHECK
def test_baseline():
    m = make_baseline().fit(np.zeros((4, 1)), ["Normal", "Critical", "Normal", "Normal"])
    assert set(m.predict(np.zeros((50, 1)))) == {"Normal"}, "the baseline should always give the most common label"

check("baseline", test_baseline)

## Step 6 · Your first model
A **pipeline** joins the steps a model needs so that they always run in the same order: fill the gaps, (scale), learn. You built one by hand in Week 5. Filling gaps inside the pipeline is one way to meet requirement 9 of the brief: a missing reading must not crash the model.

### 6a · Worked example: a straight-line model on the demo data

In [ ]:
# Worked example on the demo data: a straight-line model (logistic regression), tested on unseen patients
d = pd.read_csv(DEMO_PATH)
dX, dy = d.drop(columns=["status", "patient_id"]), d["status"]
learn, test = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(dX, dy, groups=d["patient_id"]))
example = make_pipeline(SimpleImputer(strategy="median"),     # fill each gap (sensor dropout) with the middle value
                        StandardScaler(),                     # put every column on the same scale
                        LogisticRegression(max_iter=2000))    # learn straight-line boundaries
example.fit(dX.iloc[learn], dy.iloc[learn])
for name, value in scores(dy.iloc[test], example.predict(dX.iloc[test])).items():
    print(f"{name:<18} {value:.3f}")

The straight-line model finds 75% of the Critical readings, but its balanced accuracy is only 0.63: it mixes up Warning and Normal, because a reading can be risky for several different reasons (too high **or** too low). A model that can combine conditions does better: a random forest.

### 6b · Your turn: a random forest
Replace the two `___`.

In [ ]:
def make_model():
    """Fill the gaps, then learn with a random forest: many decision trees that vote."""
    # TODO: how many trees? 200 is a good start
    learner = RandomForestClassifier(n_estimators=___, class_weight="balanced", random_state=42)
    return make_pipeline(SimpleImputer(strategy="median"), ___)   # TODO: the learner goes last

if needs("split"):
    model = attempt("model", lambda: make_model().fit(X_train, y_train))
    if STEPS["model"]:
        print("First model trained on", len(X_train), "rows")

In [ ]:
# CHECK
def test_model():
    m = make_model()
    assert isinstance(m, Pipeline), "make_model should return make_pipeline(...)"
    assert isinstance(m.steps[-1][1], RandomForestClassifier), "the forest should be the last step"
    assert m.steps[-1][1].n_estimators >= 50, "use at least 50 trees"

check("first model", test_model)

## Step 7 · Compare: did your model beat the baseline?
This cell scores both on the unseen patients, prints the measures the brief asks for, and draws the confusion matrix. You do not need to change it. **Take a screenshot of the output for your report.**

In [ ]:
if needs("baseline", "model"):
    predicted = model.predict(X_test)
    results = pd.DataFrame({"baseline": scores(y_test, baseline.predict(X_test)),
                            "first model": scores(y_test, predicted)}).round(3)
    print(results.to_string())
    key = "recall: Critical" if "recall: Critical" in results.index else "balanced accuracy"
    before, after = results.loc[key, "baseline"], results.loc[key, "first model"]
    if after > before:
        print(f"\nPASS  the model beats the baseline on {key}: {before:.3f} -> {after:.3f}")
    else:
        print(f"\nTHINK the model does not beat the baseline on {key}. That is a finding, not a failure:\n"
              "      write down why before you change anything (too little data? a missing column? labels that cannot be predicted?)")
    labels = [c for c in ORDER if c in set(y_test)] or sorted(set(y_test))
    print("\nPer label:\n" + classification_report(y_test, predicted, labels=labels, zero_division=0))
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    ConfusionMatrixDisplay.from_predictions(y_test, predicted, labels=labels, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("First model: right and wrong answers")
    plt.show()

**Think:** Read the confusion matrix. How many Critical readings did the model miss (a clinician never alerted: requirement 4)? How many Normal readings did it call Critical (a false alarm: requirement 9)? Which mistake is worse for MedLink, and does your model reach your target from Step 1?

**Your answer:**

_(double-click to type)_

## Step 8 · Save the model so the rest of your system can use it
In Week 8 an API will load this file and classify each reading as it arrives. `joblib` saves a trained model to a file and loads it back.

### 8a · Worked example

In [ ]:
# Worked example: save a tiny model, load it back, and use it
tiny = DummyClassifier(strategy="most_frequent").fit([[0], [1], [2]], ["Normal", "Critical", "Normal"])
joblib.dump(tiny, ROOT / "models" / "tiny_example.joblib")
same = joblib.load(ROOT / "models" / "tiny_example.joblib")
print("The loaded model says:", same.predict([[5]])[0])
(ROOT / "models" / "tiny_example.joblib").unlink()    # tidy up the example file

### 8b · Your turn
Replace the two `___`.

In [ ]:
MODEL_PATH = ROOT / "models" / "first_model.joblib"

def save_model(model, path):
    joblib.dump(___, ___)          # TODO: what to save, and where (see the worked example)
    return path

if needs("model"):
    if attempt("save", lambda: save_model(model, MODEL_PATH)):
        # A small description file: the Week 8 API needs the column names, the labels and one example reading
        row = X_test.iloc[0]
        info = {"target": TARGET, "labels": [str(c) for c in model.classes_], "features": list(X.columns),
                "example_row": {k: (None if pd.isna(v) else float(v)) for k, v in row.items()}}
        (ROOT / "models" / "first_model.json").write_text(json.dumps(info, indent=2))
        loaded = joblib.load(MODEL_PATH)
        print("Saved:", MODEL_PATH.relative_to(ROOT), "and models/first_model.json")
        print("Loaded it back. For the first test reading it predicts:", loaded.predict(X_test.iloc[[0]])[0],
              "| true label:", y_test.iloc[0])

In [ ]:
# CHECK
def test_saved():
    assert STEPS.get("save"), "TODO  model saved: finish Step 8b first"
    loaded = joblib.load(MODEL_PATH)
    assert list(loaded.predict(X_test)) == list(model.predict(X_test)), "the loaded model gives different answers"

check("model saved and loads", test_saved)

## Step 9 · Write it down
This cell saves your results to `docs/results/first_model.md`: evidence for Output 4 of the brief (testing with metrics). You do not need to change it.

In [ ]:
if needs("baseline", "model", "save"):
    from datetime import date
    lines = [f"# First model results ({date.today():%d %b %Y})", "",
             f"- Data: `{Path(DATA_PATH).name}` ({len(df)} rows, {X.shape[1]} feature columns)",
             f"- Predicting: `{TARGET}`",
             f"- Test set: {len(X_test)} rows, " + (f"from {df.loc[X_test.index, GROUP].nunique()} patients the model never saw"
                                                    if GROUP else "chosen at random"),
             "", "| Measure | Baseline | First model |", "|---|---|---|"]
    lines += [f"| {k} | {results.loc[k, 'baseline']} | {results.loc[k, 'first model']} |" for k in results.index]
    lines += ["", f"My target: {MY_MODEL['good_enough'] or '(not set yet: see Step 1)'}"]
    out = ROOT / "docs" / "results" / "first_model.md"
    out.write_text("\n".join(lines) + "\n")
    print("\n".join(lines))
    print("\nWritten to", out.relative_to(ROOT), "(add the confusion matrix as a screenshot in your report)")

## Step 10 · ADR 1 and three questions
Fill in `docs/adr/0001-first-model.md` with the numbers from Step 7 as evidence. Then answer:
1. Did your model beat the baseline? By how much, on which measure?
2. How many Critical readings did it miss, and how many false alarms did it raise? Which matters more for MedLink?
3. Your labels came from a rule. If the model only re-learns that rule, what has it really learned? What does that mean for your report's limitations section?

**Your answer:**

_(double-click to type)_

## Next steps
- **Commit and push** this notebook, `docs/results/first_model.md`, `models/first_model.json` and the model file (if it is under 50 MB).
- **Run the tests:** type `pytest` in the project folder. `tests/test_model.py` checks that the saved model loads, answers with one of its labels, and copes with a missing reading.
- **Compare a second approach** (Output 1 of the brief): try another model type (for example, gradient boosting or a small neural network) in `notebooks/02_...`, and record the comparison in `docs/approaches.md`.
- **Week 8:** your model goes behind an API, in a container.

## Readings or sequences?
The brief allows you to classify *"each reading (or sequence of readings)"*. A sequence can catch a slow change that no single reading shows. Add columns that summarise each patient's recent readings, then come back to Step 2:

```python
df = df.sort_values(["patient_id", "reading_time"])        # your simulator needs a time column
df["hr_mean_last5"] = df.groupby("patient_id")["heart_rate"].transform(lambda s: s.rolling(5, min_periods=1).mean())
df["spo2_change"] = df.groupby("patient_id")["spo2"].diff()
```

Windows like these are the Week 4 and Week 5 techniques. Say in your report which you chose, and why.